# Nestlé WISER DOM experiment laboratory

This notebook builds evidence before any report or presentation is generated.
It uses five canonical runtime CSV inputs and, when present, two optional
recommendation-output CSVs. Raw rows are never written to runs; only aggregate
metrics and figures are saved.

Start with the smoke profile to check installation. Set
NESTLE_EXPERIMENT_PROFILE=full for final challenge evidence. Every experiment
is checkpointed separately, so a long run can resume without repeating
completed studies.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from domopt.experiments import (
    run_challenge_experiments,
    run_remote_qpu_validation,
    write_experiment_results,
)
from domopt.hardware import (
    benchmark_qubo_batch_scoring,
    hardware_capabilities,
)
from domopt.poc import (
    POC_REFERENCE_FILENAMES,
    PocConfig,
    audit_poc_bundle,
    audit_poc_outputs,
    load_poc_problem,
    prune_pareto_candidates,
)
from domopt.visualization import (
    plot_challenge_results,
    plot_hardware_benchmark,
)


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (
            (candidate / "pyproject.toml").is_file()
            and (candidate / "src/domopt").is_dir()
        ):
            return candidate
    raise RuntimeError(
        "Run this notebook from inside the wiser-dom-optimization repository"
    )


PROJECT_ROOT = find_project_root(Path.cwd())
BUNDLE_DIR = Path(
    os.environ.get(
        "NESTLE_BUNDLE_DIR",
        PROJECT_ROOT / "data/raw/nestle_challenge",
    )
).expanduser().resolve()
PROFILE = os.environ.get(
    "NESTLE_EXPERIMENT_PROFILE", "smoke"
).strip().lower()
FORCE_RERUN = os.environ.get("NESTLE_FORCE_RERUN", "0") == "1"
ENABLE_GPU_BENCHMARK = (
    os.environ.get("DOMOPT_ENABLE_GPU_BENCHMARK", "0") == "1"
)
ENABLE_REMOTE_QPU = (
    os.environ.get("DOMOPT_ENABLE_REMOTE_QPU", "0") == "1"
)
OUTPUT_DIR = PROJECT_ROOT / "runs/challenge-study"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(
    {
        "project_root": str(PROJECT_ROOT),
        "bundle_dir": str(BUNDLE_DIR),
        "profile": PROFILE,
        "force_rerun": FORCE_RERUN,
    }
)


## 1. Runtime-input readability gate

The optimizer needs only five tables: orders, inventory planning, shipping
lanes, dock capacity, and throughput observations. The challenge PDF,
equations document, and example workbook explain the task but are not solver
inputs. The two recommendation outputs are optional audit references.

If downloads contain names such as input_order data(1).csv, run
scripts/prepare_challenge_bundle.py first. It creates stable names and excludes
numbered duplicates and macOS metadata files.


In [ ]:
file_audit = audit_poc_bundle(BUNDLE_DIR)
assert file_audit["readable"].all()
display(
    file_audit[
        ["role", "filename", "rows", "columns", "readable"]
    ]
)


## 2. Build and audit the real POC model

This step converts planning units to integer cases, identifies focus loads,
creates eligible DC/date options, protects five days of inventory, and applies
documented dock and penalty rules. Pareto pruning is delayed so its effect can
be measured rather than assumed.


In [ ]:
problem_unpruned = load_poc_problem(
    BUNDLE_DIR,
    config=PocConfig(pareto_prune=False),
    strict_bundle_audit=False,
)
problem_pruned = prune_pareto_candidates(problem_unpruned)
summary = pd.DataFrame(
    [
        {
            "variant": "unpruned",
            "orders": len(problem_unpruned.orders),
            "assignment_groups": problem_unpruned.orders[
                "assignment_group"
            ].nunique(),
            "order_lines": len(problem_unpruned.order_lines),
            "candidate_rows": len(problem_unpruned.candidates),
        },
        {
            "variant": "pareto_pruned",
            "orders": len(problem_pruned.orders),
            "assignment_groups": problem_pruned.orders[
                "assignment_group"
            ].nunique(),
            "order_lines": len(problem_pruned.order_lines),
            "candidate_rows": len(problem_pruned.candidates),
        },
    ]
)
display(summary)

reference_available = all(
    (BUNDLE_DIR / name).is_file()
    for name in POC_REFERENCE_FILENAMES.values()
)
if reference_available:
    display(
        pd.Series(
            audit_poc_outputs(BUNDLE_DIR, problem_unpruned),
            name="reference audit",
        )
    )
else:
    print(
        "Optional recommendation outputs are absent; "
        "reconciliation is skipped."
    )


## 3. Checkpointed experiment runner

Each following cell runs one study and writes its aggregate table immediately.
A rerun loads an existing checkpoint unless NESTLE_FORCE_RERUN=1. Feasibility
is checked after saving diagnostics, so an unexpected failure reports its
category instead of disappearing behind a bare assertion.


In [ ]:
experiment_frames: dict[str, pd.DataFrame] = {}


def feasible_mask(frame: pd.DataFrame) -> pd.Series:
    values = frame["feasible"]
    if values.dtype == bool:
        return values
    return values.astype(str).str.lower().isin({"true", "1"})


def run_or_load(name: str) -> pd.DataFrame:
    path = TABLE_DIR / f"{name}.csv"
    if path.is_file() and not FORCE_RERUN:
        frame = pd.read_csv(path)
        print(f"loaded checkpoint: {path.relative_to(PROJECT_ROOT)}")
    else:
        frame = run_challenge_experiments(
            problem_unpruned,
            profile=PROFILE,
            experiments=[name],
        )
        write_experiment_results(frame, path)
        print(f"wrote checkpoint: {path.relative_to(PROJECT_ROOT)}")

    invalid = frame.loc[~feasible_mask(frame)]
    if not invalid.empty:
        columns = [
            "experiment",
            "level",
            "validation_categories",
            "validation_violation_count",
        ]
        raise RuntimeError(
            "Infeasible experiment rows: "
            f"{invalid[columns].to_dict('records')}"
        )
    experiment_frames[name] = frame
    return frame


## 4. Common solver comparison

This is the central fairness test. Default, load-atomic greedy, exact MILP,
and hybrid QUBO-MILP use the same business objective and independent validator.
The exact gap shows whether optimality was proved; hybrid improvement is
measured from the strong greedy incumbent.


In [ ]:
solver_results = run_or_load("solver_comparison")
display(
    solver_results[
        [
            "method",
            "feasible",
            "objective_value",
            "case_fill_rate",
            "penalty_cost",
            "shipping_cost",
            "runtime_seconds",
            "optimality_gap",
            "hybrid_improvement",
            "maximum_qubo_variables",
        ]
    ].sort_values("objective_value", ascending=False)
)


## 5. Assignment-group size scaling

A load can contain several order records, so the true atomic decision count is
the number of assignment groups. This study scales greedy through large real
subsets, bounds hybrid to tractable neighborhoods, and runs exact MILP only
where a useful certificate is realistic. It measures runtime, objective,
candidate growth, and local QUBO width.


In [ ]:
scaling_results = run_or_load("size_scaling")
display(
    scaling_results[
        [
            "method",
            "actual_assignment_groups",
            "order_count",
            "order_line_count",
            "candidate_count",
            "maximum_qubo_variables",
            "objective_value",
            "runtime_seconds",
            "feasible",
        ]
    ].sort_values(["actual_assignment_groups", "method"])
)


## 6. Business penalty-weight sensitivity

Unmet-demand penalties encode the cost of poor service. This study uses a
fixed set of loads whose default fill is below the penalty threshold and ranks
them by active penalty exposure; otherwise a shortage-only sample can contain
zero-penalty orders and make the sweep meaningless. Scaling the penalties then
tests whether routing decisions are stable or driven by one arbitrary
coefficient. The important trade-off is fill and penalty reduction versus
extra shipping. Raw objectives across different penalty scales are not
directly comparable.


In [ ]:
business_penalty_results = run_or_load(
    "penalty_weight_sensitivity"
)
display(
    business_penalty_results[
        [
            "penalty_scale",
            "method",
            "case_fill_rate",
            "reassigned_orders",
            "penalty_cost",
            "shipping_cost",
            "runtime_seconds",
        ]
    ].sort_values(["penalty_scale", "method"])
)


## 7. QUBO penalty calibration

These are algorithmic penalties, not business costs. The one-hot multiplier
discourages selecting zero or multiple options for a load; the pair multiplier
discourages competing plans from overusing shared resources. The sweep
measures raw one-hot rate, repair burden, final improvement, and runtime so
penalties are tuned with evidence rather than guessed.


In [ ]:
qubo_penalty_results = run_or_load("qubo_penalty_sensitivity")
display(
    qubo_penalty_results[
        [
            "one_hot_penalty_multiplier",
            "pair_penalty_multiplier",
            "raw_one_hot_rate",
            "hybrid_improvement",
            "accepted_moves",
            "recourse_solves",
            "runtime_seconds",
        ]
    ].sort_values(
        ["one_hot_penalty_multiplier", "pair_penalty_multiplier"]
    )
)


## 8. Candidate-count sensitivity

Keeping more DC/date alternatives can improve the solution, but it increases
preprocessing, QUBO width, and recourse work. This experiment locates the point
where extra candidates stop paying for their computational cost.


In [ ]:
candidate_results = run_or_load("candidate_count_sensitivity")
display(
    candidate_results[
        [
            "candidate_limit",
            "method",
            "candidate_count",
            "maximum_qubo_variables",
            "objective_value",
            "case_fill_rate",
            "runtime_seconds",
        ]
    ].sort_values(["candidate_limit", "method"])
)


## 9. Inventory-shock robustness

The deterministic objective already protects projected ATP and charges shortage
penalties. A generic extra risk term would double-count risk without scenario
probabilities. Instead, this study reduces available inventory by increasing
fractions and reoptimizes under each scenario. It is the defensible starting
point for a later CVaR or robust objective if calibrated forecasts become
available.


In [ ]:
shock_results = run_or_load("inventory_shock")
display(
    shock_results[
        [
            "inventory_shock",
            "method",
            "objective_value",
            "case_fill_rate",
            "unassigned_orders",
            "penalty_cost",
            "runtime_seconds",
        ]
    ].sort_values(["inventory_shock", "method"])
)


## 10. Seed and local QUBO coefficient-noise robustness

Repeated seeds test stochastic stability. Coefficient perturbations approximate
analog/control sensitivity in the local QUBO only; they are not a physical
device-noise model. Exact quantity recourse and final validation remain
unchanged, so weak samples cannot degrade the returned incumbent.


In [ ]:
noise_results = run_or_load("quantum_seed_noise")
display(
    noise_results[
        [
            "seed",
            "coefficient_noise_relative_sigma",
            "raw_one_hot_rate",
            "hybrid_improvement",
            "accepted_moves",
            "runtime_seconds",
        ]
    ].sort_values(["coefficient_noise_relative_sigma", "seed"])
)


## 11. Pareto-pruning ablation

Pareto pruning removes a candidate only when another option is no worse in
estimated fill, value, shipping cost, and lead time, and better in at least one
dimension. Running with and without pruning checks both its speed benefit and
whether the approximation changes the best validated solution.


In [ ]:
pruning_results = run_or_load("pareto_pruning_ablation")
display(
    pruning_results[
        [
            "level",
            "candidate_count",
            "maximum_qubo_variables",
            "hybrid_improvement",
            "runtime_seconds",
            "feasible",
        ]
    ]
)


## 12. Random versus conflict-based batches

A local hybrid move is useful only if its loads interact. Conflict batching
groups loads that compete for inventory or capacity; random batching is the
control. The comparison tests whether problem-aware decomposition finds better
moves under the same QUBO and runtime limits.


In [ ]:
batch_results = run_or_load("batch_strategy_ablation")
display(
    batch_results[
        [
            "level",
            "hybrid_improvement",
            "accepted_moves",
            "maximum_qubo_variables",
            "recourse_solves",
            "runtime_seconds",
        ]
    ]
)


## 13. Sampler ablation

Random bitstrings are a deliberately weak control. Simulated annealing should
produce better-ranked local assignments if the QUBO contains useful structure.
Exact MILP recourse is identical in both cases, which isolates the sampler
proposal quality.


In [ ]:
sampler_results = run_or_load("sampler_ablation")
display(
    sampler_results[
        [
            "level",
            "raw_one_hot_rate",
            "hybrid_improvement",
            "accepted_moves",
            "runtime_seconds",
            "feasible",
        ]
    ]
)


## 14. Synthetic coordination control

Real-data subsets may not contain a case where local search beats greedy. This
independently generated control contains coupled choices that expose greedy
myopia. It tests the architecture, but it must be labeled synthetic and cannot
support a real-data or quantum-advantage claim.


In [ ]:
synthetic_results = run_or_load(
    "synthetic_coordination_control"
)
display(
    synthetic_results[
        [
            "method",
            "objective_value",
            "case_fill_rate",
            "runtime_seconds",
            "optimality_gap",
            "hybrid_improvement",
            "feasible",
        ]
    ].sort_values("objective_value", ascending=False)
)


## 15. Persist all graphics

The earlier notebook displayed plots without saving them. This cell combines
every checkpoint, writes aggregate_results.csv, saves stable PNG files under
runs/challenge-study/figures, and displays each image inline. These are evidence
plots, not a final report or slide deck.


In [ ]:
prepared_frames = [
    frame.dropna(axis=1, how="all")
    for frame in experiment_frames.values()
]
results = pd.concat(prepared_frames, ignore_index=True, sort=False)
aggregate_path = write_experiment_results(
    results, OUTPUT_DIR / "aggregate_results.csv"
)
figure_paths = plot_challenge_results(results, FIGURE_DIR)
print(
    f"wrote {len(results)} aggregate rows to "
    f"{aggregate_path.relative_to(PROJECT_ROOT)}"
)
for name, path in figure_paths.items():
    display(Markdown(f"### {name.replace('_', ' ').title()}"))
    display(Image(filename=str(path)))
print(
    f"saved {len(figure_paths)} figures in "
    f"{FIGURE_DIR.relative_to(PROJECT_ROOT)}"
)


## 16. Optional GPU crossover benchmark

The exact SciPy/HiGHS MILP is CPU-based, and the real bottlenecks are candidate
processing and exact recourse. Current local QUBOs are small, so GPU launch and
transfer overhead may exceed the work saved. This synthetic benchmark measures
only batched QUBO energy scoring and shows where an RTX-class GPU begins to
help. It does not claim end-to-end solver acceleration.

Set DOMOPT_ENABLE_GPU_BENCHMARK=1 and install the GPU extra on a CUDA machine
to run it. NVIDIA cuOpt can be evaluated later as an experimental MILP backend,
but it should be benchmarked against HiGHS before adoption.


In [ ]:
capabilities = hardware_capabilities()
display(pd.Series(capabilities, name="hardware capability"))
if ENABLE_GPU_BENCHMARK:
    hardware_results = benchmark_qubo_batch_scoring(
        include_gpu=True
    )
    hardware_path = OUTPUT_DIR / "hardware_qubo_scoring.csv"
    hardware_results.to_csv(hardware_path, index=False)
    hardware_figure = plot_hardware_benchmark(
        hardware_results,
        FIGURE_DIR / "hardware_qubo_scoring.png",
    )
    display(hardware_results)
    display(Image(filename=str(hardware_figure)))
else:
    print(
        "GPU benchmark skipped. "
        "Set DOMOPT_ENABLE_GPU_BENCHMARK=1 to run it."
    )


## 17. Optional remote D-Wave hardware validation

D-Wave is the most direct remote hardware match because the local model is
already a QUBO. This opt-in cell sends only generated synthetic coefficients,
never Nestlé values or identifiers. It records QPU calls, access timing,
chain-break fraction, raw one-hot rate, repair, exact recourse, and final
feasibility.

Install the QPU extra, configure Leap outside the notebook, and set
DOMOPT_ENABLE_REMOTE_QPU=1. A small hardware run validates integration; it does
not establish quantum advantage. IBM gate-model QAOA remains a possible second
platform, but it adds circuit depth, transpilation, and iterative runtime
overhead and is not the preferred first test for this QUBO workflow.


In [ ]:
if ENABLE_REMOTE_QPU:
    qpu_results = run_remote_qpu_validation(
        sampler="dwave-qpu",
        allow_remote=True,
        num_reads=100,
    )
    qpu_path = write_experiment_results(
        qpu_results,
        TABLE_DIR / "remote_qpu_validation.csv",
    )
    display(
        qpu_results[
            [
                "level",
                "feasible",
                "objective_value",
                "runtime_seconds",
                "qpu_calls",
                "qpu_access_time_microseconds",
                "mean_chain_break_fraction",
                "raw_one_hot_rate",
            ]
        ]
    )
    print(f"wrote {qpu_path.relative_to(PROJECT_ROOT)}")
else:
    print(
        "Remote QPU disabled. Configure D-Wave Leap and "
        "set DOMOPT_ENABLE_REMOTE_QPU=1."
    )


## 18. Interpretation guardrails and next evidence

A valid conclusion requires every returned solution to pass the independent
validator, hybrid never to fall below its greedy incumbent, and real POC
evidence to remain separate from synthetic controls. Compare business-penalty
settings through fill, penalty, shipping, and reassignment trade-offs rather
than raw objectives across different scales.

Do not add a generic risk coefficient until forecast scenarios and
probabilities can calibrate it. If those become available, the next defensible
extension is a scenario-based expected-shortfall or CVaR model compared with
the inventory-shock frontier here. Do not generate the final report or
presentation until the full profile and any approved hardware runs complete.
